In [0]:
activation_data = [
    ("U001", "2024-01-01"),
    ("U002", "2024-01-05"),
    ("U003", "2024-01-10")
]

recharge_data = [
    ("U001", "2024-01-03"),
    ("U001", "2024-01-08"),
    ("U002", "2024-01-15"),
    ("U003", "2024-01-09"),
    ("U003", "2024-01-12")
]

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date

spark = SparkSession.builder.getOrCreate()

activation_df = spark.createDataFrame(
    activation_data,
    ["user_id", "activation_date"]
).withColumn(
    "activation_date", to_date("activation_date")
)

recharge_df = spark.createDataFrame(
    recharge_data,
    ["user_id", "recharge_date"]
).withColumn(
    "recharge_date", to_date("recharge_date")
)


activation_df.show()
recharge_df.show()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
window_spec = Window.partitionBy('user_id').orderBy('recharge_date')
df_rank = recharge_df.withColumn('row_num',row_number().over(window_spec)).where(col('row_num')==1)

df_res = activation_df.join(df_rank,on="user_id",how='inner')
df_result = df_res.withColumn('diff',datediff('recharge_date','activation_date'))
df_result.show()